# 8.1 팀 프로젝트: 구조와 발표 — 미니 프로젝트 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter08_1_team_project.ipynb)

책 본문: [8.1 팀 프로젝트: 구조와 발표](https://smhanlab.com/book-ml/kor/ml2/chapter08/1.html)

이 노트북은 8.1절의 프로젝트 흐름을 **한 팀 프로젝트의 축소판**으로
한 번에 통과시킵니다:

1. **환경 고르기(M1)** — CliffWalking-v1(4×12 격자, 절벽 −100) 구조 확인.
2. **알고리즘 구현(M2)** — SARSA와 Q-learning, 갱신 규칙이 `max()` 한 줄 차이.
3. **시드 3개 실행(M3)** — 고정 \(\varepsilon=0.1\), 500 에피소드.
   학습 도중 리턴과 greedy 평가(\(\varepsilon=0\))를 나눠서.
4. **경로 시각화** — Q-learning의 최단 경로 vs SARSA의 안전 경로.
5. **\(\varepsilon\) 감쇠(M3 튜닝)** — 스케줄 하나만으로 SARSA의 고착이 사라지는지.
6. **다른 환경·다른 알고리즘** — FrozenLake에서 MC(첫방문) vs SARSA.

numpy/matplotlib/gymnasium만 씁니다 — CPU만으로 충분합니다.


In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, random
import numpy as np

# 그림 저장 위치 — 로컬 저장소 경위가 아니면 /tmp로
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):
    IMG = "/tmp"
print("그림 저장 위치:", IMG)


그림 저장 위치: /home/smhan/book-ml/kor/src/images


## 1. 환경: CliffWalking-v1 (M1 — 문제 정의)

4×12 격자입니다. 시작(S)과 목표(G)는 **아래쪽(3행)**에 있고 그 사이를
절벽이 가로막습니다 — 절벽 칸에 발을 들이면 −100을 받고 시작점으로
돌아옵니다. Chapter 6.3 실습과 같은 환경을 그대로 씁니다.
구조를 먼저 출력해서 눈으로 확인합니다:

In [2]:
import gymnasium as gym

env = gym.make("CliffWalking-v1")
n_states, n_actions = env.observation_space.n, env.action_space.n
START, GOAL = env.unwrapped.start_state_index, env.unwrapped.nS - 1
rows, cols = env.unwrapped.shape
cliff = {r*cols+c for r in range(rows) for c in range(cols) if env.unwrapped._cliff[r][c]}
print(f"격자 {rows}×{cols}, 상태 {n_states}개, 행동 {n_actions}개")
print(f"시작 S = {START} (행 {START//cols}, 열 {START%cols})")
print(f"목표 G = {GOAL} (행 {GOAL//cols}, 열 {GOAL%cols})")
print(f"절벽 {len(cliff)}개 (행 {min(c//cols for c in cliff)}, 열 {min(c%cols for c in cliff)}~{max(c%cols for c in cliff)})")
print()
for r in range(rows):
    line = ""
    for c in range(cols):
        s = r*cols+c
        line += "S" if s==START else "G" if s==GOAL else "." if s in cliff else "_"
    print(f"  {r}행: {line}")


격자 4×12, 상태 48개, 행동 4개
시작 S = 36 (행 3, 열 0)
목표 G = 47 (행 3, 열 11)
절벽 10개 (행 3, 열 1~10)

  0행: ____________
  1행: ____________
  2행: ____________
  3행: S..........G


## 2. SARSA와 Q-learning 구현 (M2 — 코드)

Chapter 6.3의 갱신 규칙을 그대로 사용합니다 — **목표값 한 줄**이
다릅니다. SARSA는 실제로 고른 다음 행동 `na`의 Q값(탐험 포함의
현실), Q-learning은 `max(Q[ns])`(항상 최선의 이상)를 씁니다.

프로젝트용 일반화 두 가지: (1) `epsilon`을 **스케줄 함수**
`eps_fn(episode)`로 받아, 고정과 감쇠를 같은 코드로 돌릴 수 있게
한다. (2) 시드 하나(0, 1, 2)로 실행합니다 — 8.1절 §"비교 프로토콜".

In [3]:
def epsilon_greedy(Q, s, epsilon, n_actions):
    if random.random() < epsilon:
        return random.randrange(n_actions)
    return max(range(n_actions), key=lambda a: Q[s][a])

def train(algo, eps_fn, n_episodes=500, alpha=0.5, gamma=1.0, seed=0):
    """algo: 'sarsa' | 'qlearn'.  eps_fn(episode) -> 그 에피소드의 epsilon.
    (Q, 에피소드별 리턴 리스트) 반환."""
    random.seed(seed)
    Q = [[0.0]*n_actions for _ in range(n_states)]
    rets = []
    for ep in range(n_episodes):
        eps = eps_fn(ep)
        s, _ = env.reset(seed=ep)
        a = epsilon_greedy(Q, s, eps, n_actions)
        tot = 0.0
        for _ in range(500):          # 에피소드 스텝 상한
            ns, r, term, trunc, _ = env.step(a)
            tot += r
            done = term or trunc
            na = epsilon_greedy(Q, ns, eps, n_actions)
            if algo == "qlearn":
                tgt = r + (gamma*max(Q[ns]) if not done else 0.0)   # off-policy
            else:
                tgt = r + (gamma*Q[ns][na] if not done else 0.0)    # on-policy
            Q[s][a] += alpha*(tgt - Q[s][a])
            s, a = ns, na
            if done:
                break
        rets.append(tot)
    return Q, rets

def greedy_rollout(Q, n_ep=200, seed=12345):
    """학습 후 평가: epsilon=0(탐욕적), 모든 알고리즘·시드에 동일한 프로토콜."""
    random.seed(seed)
    rets, steps_list = [], []
    for ep in range(n_ep):
        s, _ = env.reset(seed=ep)
        tot, steps = 0.0, 0
        for _ in range(500):
            a = max(range(n_actions), key=lambda a: Q[s][a])
            ns, r, term, trunc, _ = env.step(a)
            tot += r; steps += 1
            s = ns
            if term or trunc:
                break
        rets.append(tot); steps_list.append(steps)
    return sum(rets)/n_ep, sum(steps_list)/n_ep

def mean_last100(rets):
    return sum(rets[-100:])/100


## 3. 고정 \(\varepsilon=0.1\), 시드 0/1/2 (M3 — 시드 ≥3 실행)

8.1절 §"비교 프로토콜" 그대로: **학습 도중**(마지막 100 에피소드
평균, 탐험 포함)과 **학습 후**(greedy, 200 에피소드)를 나눠서,
시드별로 보고합니다.


In [4]:
fixed_eps = lambda ep: 0.1

print(f"{'시드':<5}{'SARSA 학습중':>14}{'SARSA greedy':>15}{'QL 학습중':>12}{'QL greedy':>12}")
results = {}
for seed in [0, 1, 2]:
    Qs, rets_s = train("sarsa", fixed_eps, seed=seed)
    Qq, rets_q = train("qlearn", fixed_eps, seed=seed)
    gs, steps_s = greedy_rollout(Qs)
    gq, steps_q = greedy_rollout(Qq)
    results[seed] = dict(Qs=Qs, rets_s=rets_s, gs=gs, Qq=Qq, rets_q=rets_q, gq=gq)
    print(f"{seed:<5}{mean_last100(rets_s):>14.1f}{gs:>15.1f}{mean_last100(rets_q):>12.1f}{gq:>12.1f}")
print()
print("(greedy −500은 500스텝 상한까지 달린 '고착' — 목표에 닿지 못하는 정책)")


시드        SARSA 학습중   SARSA greedy      QL 학습중   QL greedy
0             -31.5          -17.0       -49.6       -13.0


1             -25.6          -17.0       -51.2       -13.0
2             -23.6          -17.0       -48.6       -13.0

(greedy −500은 500스텝 상한까지 달린 '고착' — 목표에 닿지 못하는 정책)


**핵심 관찰 — 같은 실험을 두 방향으로 읽기:**

- **학습 도중**: SARSA가 Q-learning보다 낫다(절벽 추락 −100을 피하기
  때문, 6.3절의 설명).
- **greedy 평가**: Q-learning은 세 시드 모두 13스텝 최단(−13)이지만,
  **SARSA는 일부 시드에서 최종 정책이 고착(−500)**된다.
  시드 2개만 봤다면 이 고착을 못 봤을 수 있다 — §"비교 프로토콜"이
  시드 ≥3을 요구하는 이유.


## 4. greedy 정책의 경로 시각화

시드 2의 SARSA(−17, 안전 경로)와 시드 0의 Q-learning(−13, 최단 경로)의
greedy 경로를 격자에 그리면 6.3절의 "두 경로"가 눈에 보입니다 —
SARSA는 절벽에서 떨어진 위쪽을, Q-learning은 절벽 바로 옆을 갑니다:

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Patch
# 한국어 라벨을 위한 CJK 폰트 (없으면 DejaVu Sans로 fallback)
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

def greedy_path(Q, seed=0):
    random.seed(seed)
    s, _ = env.reset(seed=seed)
    path = [s]
    for _ in range(500):
        a = max(range(n_actions), key=lambda a: Q[s][a])
        s, r, term, trunc, _ = env.step(a)
        path.append(s)
        if term or trunc:
            break
    return path

path_sarsa = greedy_path(results[2]["Qs"])   # −17인 시드의 SARSA
path_q     = greedy_path(results[0]["Qq"])   # 13스텝의 Q-learning
print(f"SARSA 탐욕적 경로: {len(path_sarsa)-1} 스텝")
print(f"Q-learning 탐욕적 경로: {len(path_q)-1} 스텝")

def draw_cell(ax, s, color, edgecolor="none", zorder=2, alpha=1.0):
    r, c = s//cols, s%cols
    ax.add_patch(plt.Rectangle((c, rows-1-r), 1, 1, facecolor=color,
                               edgecolor=edgecolor, zorder=zorder, alpha=alpha))

fig, ax = plt.subplots(figsize=(10, 4))
for s in range(n_states):
    draw_cell(ax, s, "#1a1a1a" if s in cliff else "white", edgecolor="gray", zorder=1)
for s in path_sarsa[:-1]:
    if s not in cliff:
        draw_cell(ax, s, "#4a90d9", alpha=0.7, zorder=2)
for s in path_q[:-1]:
    if s not in cliff:
        draw_cell(ax, s, "#d9534f", alpha=0.9, zorder=3)
draw_cell(ax, START, "white", edgecolor="black", zorder=4)
draw_cell(ax, GOAL, "white", edgecolor="black", zorder=4)
ax.text(cols/2, 0.35, "S", ha="center", va="center", fontsize=14)
ax.text(cols/2, rows-0.35, "G", ha="center", va="center", fontsize=14)
ax.set_xlim(-0.1, cols+0.1); ax.set_ylim(-0.1, rows+0.1)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("CliffWalking: SARSA(파랑, 절벽 우회) vs Q-learning(빨강, 절벽 옆 최단)", fontsize=11)
ax.legend(handles=[
    Patch(facecolor="#4a90d9", alpha=0.6, label=f"SARSA ({len(path_sarsa)-1}스텝, 안전)"),
    Patch(facecolor="#d9534f", label=f"Q-learning ({len(path_q)-1}스텝, 최단)"),
    Patch(facecolor="#1a1a1a", label="절벽"),
], loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig(IMG + "/ch08_1_cliff_paths.svg", bbox_inches="tight")
plt.show()
print("그림 저장:", IMG + "/ch08_1_cliff_paths.svg")


SARSA 탐욕적 경로: 17 스텝
Q-learning 탐욕적 경로: 13 스텝
그림 저장: /home/smhan/book-ml/kor/src/images/ch08_1_cliff_paths.svg


## 5. \(\varepsilon\) 감쇠: 같은 500 에피소드, 같은 파라미터

고정 \(\varepsilon=0.1\)에서 SARSA의 고착(−500)이 나온다면,
**스케줄 하나만** 바꿔 500 에피소드 동안 1.0 → 0.01로 선형 감쇠시켜
보세요. 초반 탐험이 넓어지고 후반 평가에 섞이는 무작위성이 줄기
때문에, 같은 알고리즘인데도 결과가 달라집니다 — 이 실험이
"\(\varepsilon\) 스케줄도 하이퍼파라미터"라는 8.1절의 주장입니다:

In [6]:
def make_eps_schedule(n_episodes, start=1.0, end=0.01):
    def eps_fn(ep):   # 선형 감쇠
        return max(end, start + (end - start)*ep/n_episodes)
    return eps_fn

decay_eps = make_eps_schedule(500)

print(f"{'시드':<5}{'SARSA greedy':>15}{'Q-learning greedy':>19}")
for seed in [0, 1, 2]:
    Qs, _ = train("sarsa", decay_eps, seed=seed)
    Qq, _ = train("qlearn", decay_eps, seed=seed)
    gs, _ = greedy_rollout(Qs)
    gq, _ = greedy_rollout(Qq)
    print(f"{seed:<5}{gs:>15.1f}{gq:>19.1f}")


시드      SARSA greedy  Q-learning greedy


0              -17.0              -13.0


1              -17.0              -13.0


2              -17.0              -13.0


## 6. 학습 곡선: 시드 3개가 얼마나 다르고, FrozenLake에서는?

(1) 고정 \(\varepsilon=0.1\) 실험의 학습 곡선(50 에피소드 이동평균)을
시드별로 그리면, **같은 알고리즘도 시드마다 다른 곡선**이 나온다는
것이 눈에 보입니다. (2) 그 다음 FrozenLake-v1(미끄러운 얼음 — 전이
자체가 확률적)에서 **MC 제어(첫방문, Ch5.3)와 SARSA**를 1000
에피소드·시드 3개로 비교합니다 — §"비교 프로토콜"에서 말한
"환경 전이 자체의 무작위성"이 어느 쪽 알고리즘의 시드 간 편차에
더 크게 드러나는지 확인하는 것입니다:

In [7]:
eps = lambda ep: 0.1
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (algo, key, color, title) in zip(
        axes,
        [("sarsa", "rets_s", "#4a90d9", "SARSA (시드 0/1/2)"),
         ("qlearn", "rets_q", "#d9534f", "Q-learning (시드 0/1/2)")]):
    for seed in [0, 1, 2]:
        rets = results[seed][key]
        ma = np.convolve(rets, np.ones(50)/50, mode="valid")
        ax.plot(range(50, 501), ma, color=color, alpha=0.55, linewidth=1.0)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("에피소드"); ax.set_ylabel("리턴 (이동평균 50)")
    ax.grid(alpha=0.3)
fig.suptitle("CliffWalking 학습 곡선 — 고정 ε=0.1, α=0.5, γ=1.0, 500 에피소드", fontsize=12)
fig.tight_layout()
fig.savefig(IMG + "/ch08_1_seeds_curves.svg", bbox_inches="tight")
plt.show()
print("그림 저장:", IMG + "/ch08_1_seeds_curves.svg")


그림 저장: /home/smhan/book-ml/kor/src/images/ch08_1_seeds_curves.svg


In [8]:
# --- FrozenLake-v1: 첫방문 MC 제어 vs SARSA (1000 에피소드, 시드 3개) ---
def train_frozen(algo, seed, n_episodes=1000, alpha=0.5, gamma=0.99):
    """algo: 'mc' (첫방문, Ch5.3) | 'sarsa'.  epsilon 스케줄 1.0 -> 0.1."""
    rng = random.Random(seed)
    fenv = gym.make("FrozenLake-v1")
    nS = fenv.observation_space.n
    Q = {s: [0.0]*4 for s in range(nS)}
    rets = []
    for ep in range(n_episodes):
        eps = max(0.1, 1.0 - 0.9*ep/n_episodes)
        s, _ = fenv.reset(seed=rng.randrange(2**31))
        G, done, seen = 0.0, False, set()
        while not done:
            a = rng.randrange(4) if rng.random() < eps else max(range(4), key=lambda a: Q[s][a])
            ns, r, term, trunc, _ = fenv.step(a)
            done = term or trunc
            if algo == "mc":
                if (s, a) not in seen:
                    seen.add((s, a))
            else:  # sarsa
                if not done:
                    ap = rng.randrange(4) if rng.random() < eps else max(range(4), key=lambda a: Q[ns][a])
                    tgt = r + gamma*Q[ns][ap]
                else:
                    tgt = r
                Q[s][a] += alpha*(tgt - Q[s][a])
            G += r; s = ns
        if algo == "mc":
            for (ss, aa) in seen:
                Q[ss][aa] += alpha*(G - Q[ss][aa])
        rets.append(G)
    fenv.close()
    return Q, rets

def eval_frozen(Q, n=200, seed=12345):
    rng = random.Random(seed)
    fenv = gym.make("FrozenLake-v1")
    tot = 0.0
    for _ in range(n):
        s, _ = fenv.reset(seed=rng.randrange(2**31))
        G, done = 0.0, False
        while not done:
            a = max(range(4), key=lambda a: Q[s][a])
            s, r, t, tr, _ = fenv.step(a); G += r; done = t or tr
        tot += G
    fenv.close()
    return tot/n

print(f"{'시드':<5}{'MC 학습중':>12}{'MC greedy':>12}{'SARSA 학습중':>14}{'SARSA greedy':>14}")
for seed in [0, 1, 2]:
    Qm, rm = train_frozen("mc", seed)
    Qs, rs = train_frozen("sarsa", seed)
    em = eval_frozen(Qm)
    es = eval_frozen(Qs)
    print(f"{seed:<5}{np.mean(rm[-200:]):>12.3f}{em:>12.3f}{np.mean(rs[-200:]):>14.3f}{es:>14.3f}")
print()
print("(FrozenLake는 성공=+1만 있어 리턴이 0~1 — greedy 평가가 '성공 확률' 그 자체)")


시드         MC 학습중   MC greedy     SARSA 학습중  SARSA greedy


0           0.065       0.125         0.105         0.245


1           0.050       0.090         0.115         0.730


2           0.005       0.045         0.120         0.525

(FrozenLake는 성공=+1만 있어 리턴이 0~1 — greedy 평가가 '성공 확률' 그 자체)


## 정리: 코드 → 보고서 → 리뷰로

| 이 노트북 단계 | 보고서 절 (8.1 §보고서 구조) | 8.2 체크리스트 항목 |
|---|---|---|
| 1. CliffWalking 구조 | 문제 정의(MDP 5요소) | 환경 설명 |
| 2. SARSA/Q-learning 구현 | 알고리즘 선택과 이유 | 알고리즘 선택 |
| 3. 시드 3개 + greedy 평가 | 실험 프로토콜 | 결과의 정직성 |
| 4~5. 경로·감쇠 실험 | 수식적 정당화(6.3절 on/off-policy) | 수식적 정당화 |
| 6. FrozenLake MC vs SARSA | 잘 안 된 부분(시드 간 편차)의 진단 | 결과의 정직성 |

**같은 절차를 세 가지 언어(코드, 보고서, 리뷰)로 말할 수 있어야**
팀 프로젝트가 끝난 것입니다 — 다음 수업(8.2)에서는 이 표를 반대로
읽으며 다른 팀의 발표를 리뷰합니다.
